# Ch 28. Data Parallelism — Solutions

> This notebook contains reproducible solutions to all five exercises.


## Problem 1

**Problem**: In a 4-GPU DDP simulation, show that gradients are identical after each GPU trains on different data.

### Solution

After computing local gradients $g_r$, All-Reduce copies the mean $\bar g=N^{-1}\sum_r g_r$ to every replica. Thus every rank has elementwise-identical gradient tensors immediately after synchronization.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import torch
g=torch.Generator().manual_seed(28); w=torch.randn(3,requires_grad=True,generator=g); batches=[]; local=[]
for _ in range(4):
    x=torch.randn(5,3,generator=g); y=torch.randn(5,generator=g); batches.append((x,y))
    local.append(torch.autograd.grad(((x@w-y)**2).mean(),w)[0])
allreduce_average=torch.stack(local).mean(0)
full_x=torch.cat([x for x,_ in batches]); full_y=torch.cat([y for _,y in batches])
single_process_reference=torch.autograd.grad(((full_x@w-full_y)**2).mean(),w)[0]
max_error=float((allreduce_average-single_process_reference).abs().max())
assert torch.allclose(allreduce_average,single_process_reference,atol=1e-6,rtol=1e-6)
{'allreduce_average':allreduce_average,'full_batch_reference':single_process_reference,'max_abs_error':max_error}


## Problem 2

**Problem**: Train with effective batch sizes 32, 64, 128, and 256 and compare convergence.

### Solution

Change only batch size while fixing total examples, optimizer, seed, and epochs. Compare stepwise loss on the same validation set, not only training loss, to separate gradient noise from update-count effects.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
rng=np.random.default_rng(2); true=np.array([2.,-1.]); X=rng.normal(size=(256,2)); y=X@true
def train(bs):
 w=np.zeros(2)
 for _ in range(20):
  for i in range(0,256,bs): w-=.1*(2/bs)*X[i:i+bs].T@(X[i:i+bs]@w-y[i:i+bs])
 return np.mean((X@w-y)**2)
loss={b:train(b) for b in [32,64,128,256]}; assert all(np.isfinite(list(loss.values()))); loss


## Problem 3

**Problem**: Explain why All-Reduce communication volume is $O(\|\theta\|)$.

### Solution

Ring All-Reduce sends about $(N-1)P/N$ elements in each of reduce-scatter and all-gather for tensor size $P=|\theta|$. Total is below $2P$, hence $O(P)$ per rank.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
P=10_000_000; workers=[2,4,8]; volume={n:2*(n-1)/n*P for n in workers}
assert all(v<2*P for v in volume.values()); volume


## Problem 4

**Problem**: Explain why overlapping communication and computation improves speed.

### Solution

Serial time is $C+M$, whereas the ideal overlapped lower bound is $\max(C,M)$. Implementations launch asynchronous communication as soon as backward buckets are ready, hiding part of it from the critical path.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
compute,comm=8.,5.; serial=compute+comm; overlapped=max(compute,comm)
assert overlapped<serial; {'serial':serial,'overlapped_ideal':overlapped,'speedup':serial/overlapped}


## Problem 5

**Problem**: Explain why LARS helps with large batches.

### Solution

With large batches, weight-to-gradient norm ratios may differ greatly across layers. LARS uses $\eta_l=\eta\|w_l\|/(\|g_l\|+\lambda\|w_l\|)$ to control each layer’s relative update size.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
theta=np.array([100.,1.]); grad=np.array([1.,1.]); eta=.1
trust=eta*np.linalg.norm(theta)/(np.linalg.norm(grad)+1e-9)
assert trust>eta; {'global_lr':eta,'layer_scaled_lr':trust}
